In [0]:
%run ../00_common/data_utils

In [0]:
batch_id =  dbutils.widgets.get("batch_id")
print(f"batch_id: {batch_id}")

In [0]:
def main(batch_id):
    source_table = f"{get_env_config('silver_touchpoint_parsed_database')}.t_touchpoint_sapbi_source"
    target_table = f"{get_env_config('silver_touchpoint_parsed_database')}.t_touchpoint_sapbi"

    source_df_for_batch = (
        spark.table(source_table)
        .where(F.col("BATCH_ID") == batch_id)
        .where(~((F.upper(F.col("source_type")) == "APAC") & (F.upper(F.col("MarketCode")) == "Korea")))
    )

    window_spec = Window.partitionBy("CustomerNumber", "DivisionCode").orderBy(
        F.to_date(F.col("file_date"), "yyyy-MM-dd").desc(),
        F.col("MarketCode").desc()
    )

    source_for_merge = (
        source_df_for_batch
        .withColumn("rn", F.row_number().over(window_spec))
        .where(F.col("rn") == 1)
        .drop("rn")
    )

    target_delta = DeltaTable.forName(spark, target_table)
    merge_condition = "target.CustomerNumber = source.CustomerNumber AND target.DivisionCode = source.DivisionCode"

    (target_delta.alias("target")
        .merge(source_for_merge.alias("source"), merge_condition)
        .whenMatchedUpdate(set={
            "MarketCode": "source.MarketCode",
            "SalesOrganisation": "source.SalesOrganisation",
            "BusinessType": "source.BusinessType",
            "CustomerName": "source.CustomerName",
            "RetailerOnlineDoor": "source.RetailerOnlineDoor",
            "CustomerGroup": "source.CustomerGroup",
            "ReportingGroup_Global_Code": "source.ReportingGroup_Global_Code",
            "ReportingGroup_Global_Name": "source.ReportingGroup_Global_Name",
            "ReportingGroup_Regional_Code": "source.ReportingGroup_Regional_Code",
            "ReportingGroup_Regional_Name": "source.ReportingGroup_Regional_Name",
            "ReportingGroup_Affiliate_Code": "source.ReportingGroup_Affiliate_Code",
            "ReportingGroup_Affiliate_Name": "source.ReportingGroup_Affiliate_Name",
            "Retailer": "source.Retailer",
            "Region": "source.Region",
            "City": "source.City",
            "Filename_SAPBI": "source.Filename_SAPBI",
            "UPDATE_DT": "source.UPDATE_DT",
            "BATCH_ID": "source.BATCH_ID",
            "source_type": "source.source_type",
            "file_date": "source.file_date"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
with StepLogger("t_touchpoint_sapbi", "02-2", "touchpoint", task_id=batch_id) as logger:
    main(batch_id)